[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)][def]

[def]: https://colab.research.google.com/github/LEE-Sanghoon/MNT2009/blob/main/AI-SVM.ipynb

# 서포트 벡터 머신

SVM(Support Vector Manine)은 분류 문제에 사용하는 머신러닝 알고리즘 이다. 핵심 아이디어는 데이터를 나누는 가장 좋은 경계선을 찾는 것이다. 두 종류의 데이터를 분류한다고 하면, 가장 좋은 경계선은 두 클래스 사이의 거리 간격이 가장 넓은 선이고, 이것이 둘 사이의 가운데 선이다.  이 가운데 선을 **결정 경계(decision boundary)** 이라고 한다. 그리고 결정 경계와 가장 가까운 데이터까지의 거리가 **마진(margin)** 이다. SVM은 마진을 최대한 크게 만드는 방향으로 학습 한다. 

Support Vector는 경정 경계에 가장 가까이 있는 데이터 포인트들이다. 이 데이터들이 경계 위치를 결정하기 때문에 "support" vector라고 부른다. 즉, SVM은 모든 데이터 보다도 경계 근처의 중요한 데이터들에 집중하는 모델이다.

## Scikit-learn 을 이용한 SVM 분류기

- sklearn.svm.SVC의 주요 kernel
  
  ```python
  from sklearn.svm import SVC
  ```

  | kernel | 설명 |
  | --- | --- |
  | `linear` | 선형 결정 경계 사용 |
  | `poly` | 다항식 커널 사용 |
  | `rbf` | RBF, 가우시안 커널 사용 |
  | `sigmoid` | 시그모이드 커널 사용 |
  | `precomputed` | 미리 계산한 커널 행렬 사용 |


- linear
  - 특징 수가 많고, 데이터가 어느 정도 선형적으로 구분될 때 사용
  - MNIST 처럼 784차원 데이터에도 사용할 수 있지만, 데이터가 많으면 느릴 수 있음
  - 선형 SVM만 사용할 때 `SVC(kernel="linear")` 보다 `LinearSVC`가 더 빠른 경우가 많음
  ```python
  from sklearn.svm import SVC, LinearSVC
  
  model1 = SVC(kernel="linear")     # libsvm 기반 구현
  model2 = LinearSVC()              # liblinear 기반 구현

  model1.fit(...)
  model2.fit(...)
  ```

- poly
  - 다항식 형태의 복잡한 결정 경계를 사용
  - 주요 옵션 (`degree=3`, `coef0=0.0`, `gamma="scale"`)을 설정할 수 있음
  ```python
  model = SVC(kernel="poly", degree=2)
  ```

- rbf
  - 가장 많이 쓰이는 기본 커널 (SVC의 기본값이 `kernel="rbf" 임)
  - 비선형 분류에 강함
  - 주요 옵션 (`C=1.0`, `gamma="scale"`)을 설정할 수 있음
    - MNIST 같은 이미지 분류에서 다음과 같이 사용 할 수 있음
  ```python
  model = SVC(kernel="rbf", C=10, gamma="scale")
  ```

**참고**
- `degree` : kernel="poly" 일 때 의미가 있음. 다항식의 차수에 해당
  - 작음 : 단순한 곡선 경계, 과소적합 가능성 증가, 학습 속도가 상대적으로 빠름
  - 큼 : 복잡한 경계, 훈련 데이터에 더 민감, 과적합 가능성 증가, 학습 속도가 느려질 수 있음
- `C` : 규제 설정, C=1.0 은 특별한 규제도 아닌 기본적인 균형값
  - C가 작으면 오류 허용, 마진 넓게, 모델 단순, 과소적합 가능성 증가
  - C가 클수록 오류에 강한 벌점, 훈련 데이터에 더 민감, 모델 복잡, 과적합 가능성 증가
- `gamma` : 한 데이터 포인트가 영향을 미치는 범위
  - 작은 gamma : 부드러운 결정 경계, 멀리 있는 데이터까지 비슷하게 봄, 모델이 단순해짐, 과소적합 가능성 증가
  - 큰 gamma : 복잡한 결정 경계, 가까운 데이터에만 강하게 반응, 결정 경계가 복잡해짐, 훈련 데이터에 민감해짐, 과적합 가능성 증가

## [실습] MNIST 데이터셋으로 SVM 분류 모델을 학습하기


- Download Full MNIST (70,000 images, 28x28 pixels) 

  수업 시간 중에는 Toy Digits Dataset을 사용하자.
  ```python
  from sklearn.datasets import load_digits
  mnist = load_digits()
  ```

In [6]:
from sklearn.datasets import fetch_openml

mnist = fetch_openml('mnist_784', version=1, cache=True, as_frame=False)

- Split the dataset into training and test sets

In [7]:
import numpy as np

X = mnist['data']
y = mnist['target'].astype(np.uint8)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split (
    X, y,
    test_size = 0.2,        # 전체 데이터 중 20%를 테스트셋으로 사용
    random_state = 42       # 랜덤 분할 결과를 고정. (42는 관습적인 값)
)

- Train the model

In [9]:
from sklearn.svm import SVC

model = SVC(C=10, gamma="scale")
model.fit(X_train, y_train)

SVC(C=10)

In [ ]:
from sklearn.svm import LinearSVC

model = LinearSVC(random_state=42)
model.fit(X_train, y_train)

LinearSVC(random_state=42)

- Test the model

In [10]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)
print("Test accuracy: ", accuracy_score(y_test, y_pred))

Test accuracy:  0.9822857142857143


- 틀린 결과 확인하기
  - y_test와 y_pred가 다른 때만 골라서 출력(확인) 
  ```python
  import numpy as np
  wrong_idx = np.where(y_test != y_pred)[0]
  ```
  - MNIST
  ```python
  import matplotlib.pyplot as plt
  for i in wrong_idx[:10]:  # 틀린 것 중 10개만 출력
    plt.imshow(X_test[i].reshape(28, 28), cmap="gray")
    plt.axis("off")
    plt.show()
  ```

In [ ]:
import numpy as np
wrong_idx = np.where(y_test != y_pred)[0]

print("틀린 개수: ", len(wrong_idx))

틀린 개수:  248


In [ ]:
import matplotlib.pyplot as plt

for i in wrong_idx[:10]:  # 틀린 것 중 10개만 출력
    plt.imshow(X_test[i].reshape(28, 28), cmap="gray")
    plt.axis("off")
    plt.show()